In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Setup Chrome driver automatically
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Open the website
driver.get("https://cspdcl.co.in/cseb/frmPowerOutageInfo.aspx")

# Wait for page to load
time.sleep(5)

# Function to extract table data
def extract_table_data(outage_type):
    try:
        time.sleep(2)  # Wait for table to load
        from io import StringIO
        html_source = StringIO(driver.page_source)
        tables = pd.read_html(html_source)
        if len(tables) > 7:  # Make sure table 7 exists
            df = tables[7]  # Use table index 7 (the correct outage table)
            df['Outage_Type'] = outage_type  # Add column to identify the source
            return df
        else:
            print(f"Warning: Only {len(tables)} tables found, expected at least 8")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error extracting table: {e}")
        return pd.DataFrame()

# Dictionary to store all dataframes
all_data = {}

# 1. Extract Live Outages (default view)
print("Extracting Live Outages...")
all_data['Live_Outages'] = extract_table_data('Live Outages')

# 2. Click "All Outages(Today)" radio button
print("Extracting All Outages (Today)...")
try:
    # Find and click the radio button using multiple strategies
    wait = WebDriverWait(driver, 10)
    
    # Try different XPath strategies
    radio_found = False
    strategies = [
        "//input[@type='radio' and @value='rdoAllOutagesToday']",
        "//input[@type='radio' and contains(@name, 'AllOutagesToday')]",
        "//label[contains(text(), 'All Outages(Today)')]/preceding-sibling::input[@type='radio']",
        "//label[contains(text(), 'All Outages(Today)')]/input[@type='radio']",
        "//input[@id='rdoAllOutagesToday']",
    ]
    
    for strategy in strategies:
        try:
            today_radio = driver.find_element(By.XPATH, strategy)
            driver.execute_script("arguments[0].click();", today_radio)
            radio_found = True
            print("  Radio button clicked successfully")
            break
        except:
            continue
    
    if not radio_found:
        print("  Could not find radio button, trying all radio buttons...")
        radios = driver.find_elements(By.XPATH, "//input[@type='radio']")
        if len(radios) >= 2:
            driver.execute_script("arguments[0].click();", radios[1])  # Try 2nd radio button
            radio_found = True
    
    if radio_found:
        all_data['All_Outages_Today'] = extract_table_data('All Outages (Today)')
        print("Successfully extracted Today's data")
    else:
        all_data['All_Outages_Today'] = pd.DataFrame()
        print("Failed to click radio button")
except Exception as e:
    print(f"Error with Today radio: {e}")
    all_data['All_Outages_Today'] = pd.DataFrame()

# 3. Click "All Outages(Tomorrow)" radio button
print("Extracting All Outages (Tomorrow)...")
try:
    # Find and click the radio button using multiple strategies
    radio_found = False
    strategies = [
        "//input[@type='radio' and @value='rdoAllOutagesTomorrow']",
        "//input[@type='radio' and contains(@name, 'AllOutagesTomorrow')]",
        "//label[contains(text(), 'All Outages(Tomorrow)')]/preceding-sibling::input[@type='radio']",
        "//label[contains(text(), 'All Outages(Tomorrow)')]/input[@type='radio']",
        "//input[@id='rdoAllOutagesTomorrow']",
    ]
    
    for strategy in strategies:
        try:
            tomorrow_radio = driver.find_element(By.XPATH, strategy)
            driver.execute_script("arguments[0].click();", tomorrow_radio)
            radio_found = True
            print("  Radio button clicked successfully")
            break
        except:
            continue
    
    if not radio_found:
        print("  Could not find radio button, trying all radio buttons...")
        radios = driver.find_elements(By.XPATH, "//input[@type='radio']")
        if len(radios) >= 3:
            driver.execute_script("arguments[0].click();", radios[2])  # Try 3rd radio button
            radio_found = True
    
    if radio_found:
        all_data['All_Outages_Tomorrow'] = extract_table_data('All Outages (Tomorrow)')
        print("Successfully extracted Tomorrow's data")
    else:
        all_data['All_Outages_Tomorrow'] = pd.DataFrame()
        print("Failed to click radio button")
except Exception as e:
    print(f"Error with Tomorrow radio: {e}")
    all_data['All_Outages_Tomorrow'] = pd.DataFrame()

# Close the browser
driver.quit()

# Display all extracted data
print("\n" + "="*80)
print("LIVE OUTAGES")
print("="*80)
if not all_data['Live_Outages'].empty:
    print(all_data['Live_Outages'])
else:
    print("No data available")

print("\n" + "="*80)
print("ALL OUTAGES (TODAY)")
print("="*80)
if not all_data['All_Outages_Today'].empty:
    print(all_data['All_Outages_Today'])
else:
    print("No data available")

print("\n" + "="*80)
print("ALL OUTAGES (TOMORROW)")
print("="*80)
if not all_data['All_Outages_Tomorrow'].empty:
    print(all_data['All_Outages_Tomorrow'])
else:
    print("No data available")

# Combine all data into one DataFrame (only non-empty ones)
dfs_to_combine = [df for df in [all_data['Live_Outages'], 
                                 all_data['All_Outages_Today'], 
                                 all_data['All_Outages_Tomorrow']] if not df.empty]

if dfs_to_combine:
    combined_df = pd.concat(dfs_to_combine, ignore_index=True)
    print("\n" + "="*80)
    print("COMBINED DATA (All three categories)")
    print("="*80)
    print(combined_df)
else:
    combined_df = pd.DataFrame()
    print("\nNo data to combine")

Extracting Live Outages...
Extracting All Outages (Today)...
  Radio button clicked successfully
Successfully extracted Today's data
Extracting All Outages (Tomorrow)...
  Radio button clicked successfully
Successfully extracted Tomorrow's data

LIVE OUTAGES
   SN Name of Town Maintenance activity Scheduled for Outage Affected Area  \
0   1         DURG                  11KV PUSHP VATIKA              BAGHERA   

  Outage Area Outage Start Date Outage Start Time Outage End Date (Expected)  \
0         ...        2026-03-14          03:20:00                 2026-03-14   

  Outage End Time (Expected)   Outage_Type  
0                   04:00:00  Live Outages  

ALL OUTAGES (TODAY)
    SN   Name of Town Maintenance activity Scheduled for  \
0    1            NaN                          11KV KONI   
1    2            NaN                      11KV KOSAMDIH   
2    3            NaN                  11KV MASTURI TOWN   
3    4            NaN                         11KV RISDA   
4    5   JAI

In [3]:
# Save data to CSV files for your project
saved_files = []

if not all_data['Live_Outages'].empty:
    all_data['Live_Outages'].to_csv('live_outages.csv', index=False)
    saved_files.append('live_outages.csv')

if not all_data['All_Outages_Today'].empty:
    all_data['All_Outages_Today'].to_csv('all_outages_today.csv', index=False)
    saved_files.append('all_outages_today.csv')

if not all_data['All_Outages_Tomorrow'].empty:
    all_data['All_Outages_Tomorrow'].to_csv('all_outages_tomorrow.csv', index=False)
    saved_files.append('all_outages_tomorrow.csv')

if not combined_df.empty:
    combined_df.to_csv('combined_outages.csv', index=False)
    saved_files.append('combined_outages.csv')

if saved_files:
    print("Data saved successfully!")
    for file in saved_files:
        print(f"- {file}")
else:
    print("No data was saved (all dataframes were empty)")

Data saved successfully!
- live_outages.csv
- all_outages_today.csv
- all_outages_tomorrow.csv
- combined_outages.csv


In [4]:
# Quick check of what data we extracted
print("Live Outages shape:", all_data['Live_Outages'].shape)
print("\nLive Outages columns:", all_data['Live_Outages'].columns.tolist())
print("\nFirst few rows:")
print(all_data['Live_Outages'].head())
print("\n" + "="*80)
print("All Outages Today shape:", all_data['All_Outages_Today'].shape)
print("\nAll Outages Tomorrow shape:", all_data['All_Outages_Tomorrow'].shape)

Live Outages shape: (1, 10)

Live Outages columns: ['SN', 'Name of Town', 'Maintenance activity Scheduled for', 'Outage Affected Area', 'Outage Area', 'Outage Start Date', 'Outage Start Time', 'Outage End Date (Expected)', 'Outage End Time (Expected)', 'Outage_Type']

First few rows:
   SN Name of Town Maintenance activity Scheduled for Outage Affected Area  \
0   1         DURG                  11KV PUSHP VATIKA              BAGHERA   

  Outage Area Outage Start Date Outage Start Time Outage End Date (Expected)  \
0         ...        2026-03-14          03:20:00                 2026-03-14   

  Outage End Time (Expected)   Outage_Type  
0                   04:00:00  Live Outages  

All Outages Today shape: (41, 10)

All Outages Tomorrow shape: (1, 7)


In [6]:
# Let's inspect ALL tables on the page to find the right one
from io import StringIO

# Re-open browser to check
driver2 = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver2.get("https://cspdcl.co.in/cseb/frmPowerOutageInfo.aspx")
time.sleep(5)

# Get all tables
all_tables = pd.read_html(StringIO(driver2.page_source))

print(f"Found {len(all_tables)} tables on the page\n")

# Check each table
for i, table in enumerate(all_tables):
    print(f"\n{'='*80}")
    print(f"TABLE {i}: Shape = {table.shape}")
    print(f"{'='*80}")
    print("Columns:", table.columns.tolist())
    print("\nFirst 3 rows:")
    print(table.head(3))
    print("\n")

driver2.quit()

# This will help us identify which table index has the actual outage data

Found 12 tables on the page


TABLE 0: Shape = (35, 9)
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8]

First 3 rows:
                                                   0  \
0  //<![CDATA[ Sys.WebForms.PageRequestManager._i...   
1  Chhattisgarh State Power Distribution Company ...   
2                                                NaN   

                                                   1  \
0  //<![CDATA[ Sys.WebForms.PageRequestManager._i...   
1  Chhattisgarh State Power Distribution Company ...   
2  Chhattisgarh State Power Distribution Company ...   

                                                   2    3    4    5    6    7  \
0  //<![CDATA[ Sys.WebForms.PageRequestManager._i...  NaN  NaN  NaN  NaN  NaN   
1  Chhattisgarh State Power Distribution Company ...  NaN  NaN  NaN  NaN  NaN   
2                                                NaN  NaN  NaN  NaN  NaN  NaN   

     8  
0  NaN  
1  NaN  
2  NaN  



TABLE 1: Shape = (1, 3)
Columns: [0, 1, 2]

First 3 rows:
    0                

In [2]:
# Find the correct outage table by checking for key columns
from io import StringIO
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

driver3 = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver3.get("https://cspdcl.co.in/cseb/frmPowerOutageInfo.aspx")
time.sleep(5)

all_tables = pd.read_html(StringIO(driver3.page_source))

print(f"Analyzing {len(all_tables)} tables to find the outage data...\n")

# Look for tables that might contain outage data
# The outage table should have columns related to town, area, date, time
keywords = ['town', 'area', 'outage', 'date', 'time', 'maintenance', 'sn', 'serial']

for i, table in enumerate(all_tables):
    # Check if any column name contains our keywords
    col_str = ' '.join([str(col).lower() for col in table.columns])
    
    # Check first few rows for content
    has_data = table.shape[0] > 0 and not table.empty
    
    # Count how many non-NaN values
    non_nan_ratio = table.notna().sum().sum() / (table.shape[0] * table.shape[1]) if table.size > 0 else 0
    
    print(f"TABLE {i}:")
    print(f"  Shape: {table.shape}")
    print(f"  Columns: {table.columns.tolist()[:5]}{'...' if len(table.columns) > 5 else ''}")
    print(f"  Non-NaN ratio: {non_nan_ratio:.2%}")
    
    # Check if this looks like the outage table
    if any(keyword in col_str for keyword in keywords):
        print(f"  ✓ POTENTIAL OUTAGE TABLE (contains keywords)")
        print(f"  First row sample:")
        print(f"  {table.iloc[0].to_dict() if len(table) > 0 else 'Empty'}")
    
    # Also check tables with 8-10 columns (typical for outage table)
    if 8 <= table.shape[1] <= 10 and table.shape[0] > 1 and non_nan_ratio > 0.3:
        print(f"  ✓ LOOKS PROMISING (good dimensions and data)")
        print(f"  Sample row 1:")
        if len(table) > 1:
            print(f"  {table.iloc[1].to_dict()}")
    
    print()

driver3.quit()

print("\n" + "="*80)
print("Based on the analysis above, identify which TABLE number has the outage data")
print("="*80)

Analyzing 12 tables to find the outage data...

TABLE 0:
  Shape: (35, 9)
  Columns: [0, 1, 2, 3, 4]...
  Non-NaN ratio: 17.14%

TABLE 1:
  Shape: (1, 3)
  Columns: [0, 1, 2]
  Non-NaN ratio: 33.33%

TABLE 2:
  Shape: (20, 9)
  Columns: [0, 1, 2, 3, 4]...
  Non-NaN ratio: 23.33%

TABLE 3:
  Shape: (13, 9)
  Columns: [0, 1, 2, 3, 4]...
  Non-NaN ratio: 24.79%

TABLE 4:
  Shape: (1, 2)
  Columns: [0, 1]
  Non-NaN ratio: 100.00%

TABLE 5:
  Shape: (9, 9)
  Columns: [0, 1, 2, 3, 4]...
  Non-NaN ratio: 30.86%
  ✓ LOOKS PROMISING (good dimensions and data)
  Sample row 1:
  {0: 'Live OutagesAll Outages(Today)All Outages(Tomorrow)', 1: nan, 2: nan, 3: nan, 4: nan, 5: nan, 6: nan, 7: nan, 8: nan}

TABLE 6:
  Shape: (1, 3)
  Columns: [0, 1, 2]
  Non-NaN ratio: 100.00%

TABLE 7:
  Shape: (1, 9)
  Columns: ['SN', 'Name of Town', 'Maintenance activity Scheduled for', 'Outage Affected Area', 'Outage Area']...
  Non-NaN ratio: 100.00%
  ✓ POTENTIAL OUTAGE TABLE (contains keywords)
  First row sample

In [3]:
# Debug: Find the exact radio button selectors on the page
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

driver_debug = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver_debug.get("https://cspdcl.co.in/cseb/frmPowerOutageInfo.aspx")
time.sleep(5)

print("Finding all radio buttons on the page...\n")

# Find all radio buttons
radios = driver_debug.find_elements(By.XPATH, "//input[@type='radio']")

print(f"Found {len(radios)} radio buttons:\n")

for i, radio in enumerate(radios):
    try:
        radio_id = radio.get_attribute('id')
        radio_name = radio.get_attribute('name')
        radio_value = radio.get_attribute('value')
        radio_checked = radio.get_attribute('checked')
        
        # Try to find associated label
        try:
            label = driver_debug.find_element(By.XPATH, f"//label[@for='{radio_id}']")
            label_text = label.text
        except:
            label_text = "No label"
        
        print(f"Radio {i}:")
        print(f"  ID: {radio_id}")
        print(f"  Name: {radio_name}")
        print(f"  Value: {radio_value}")
        print(f"  Checked: {radio_checked}")
        print(f"  Label: {label_text}")
        print()
    except Exception as e:
        print(f"Radio {i}: Error reading attributes - {e}")
        print()

driver_debug.quit()

print("\n" + "="*80)
print("Use the radio button details above to update Cell 1")
print("="*80)

Finding all radio buttons on the page...

Found 5 radio buttons:

Radio 0:
  ID: RadioButtonList1_0
  Name: ctl00$RadioButtonList1
  Value: hi-IN
  Checked: None
  Label: Hindi

Radio 1:
  ID: RadioButtonList1_1
  Name: ctl00$RadioButtonList1
  Value: en-US
  Checked: true
  Label: English

Radio 2:
  ID: MainContent_RdOutageDate_0
  Name: ctl00$MainContent$RdOutageDate
  Value: 2026-03-12 18:35:40
  Checked: true
  Label: Live Outages

Radio 3:
  ID: MainContent_RdOutageDate_1
  Name: ctl00$MainContent$RdOutageDate
  Value: 2026-03-12
  Checked: None
  Label: All Outages(Today)

Radio 4:
  ID: MainContent_RdOutageDate_2
  Name: ctl00$MainContent$RdOutageDate
  Value: 2026-03-13
  Checked: None
  Label: All Outages(Tomorrow)


Use the radio button details above to update Cell 1


## For Real-Time Data Collection Without API

Since CSPDCL doesn't provide an API, here are your options for real-time data:

### 1. **Scheduled Web Scraping (Recommended)**
- Run this script automatically at regular intervals (e.g., every 15-30 minutes)
- Use Windows Task Scheduler or Python's `schedule` library
- Store results in a database with timestamps

### 2. **Continuous Monitoring Script**
- Create a loop that scrapes data every X minutes
- Add timestamp to each record
- Detect changes and send alerts

### 3. **For Your Project: Predicting Hospital/School Status**
You'll need to:
1. Identify hospitals and schools in the "Outage Area" column
2. Cross-reference with your facility database
3. Check timing columns to predict when they'll be affected
4. Update status in real-time based on current time vs outage schedule

In [5]:
# Example: Real-Time Monitoring Script
# Run this continuously to monitor power outages

import schedule
import time
from datetime import datetime

def scrape_and_update():
    """
    Function to scrape data and update your database/files
    This will run at scheduled intervals
    """
    print(f"\n{'='*80}")
    print(f"Scraping at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}")
    
    # Your scraping code here (same as Cell 1)
    # After scraping, you can:
    # 1. Save to database with timestamp
    # 2. Compare with previous data to detect changes
    # 3. Send alerts if hospitals/schools are affected
    # 4. Update dashboard or API
    
    # Add timestamp column
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Example: Save with timestamp
    filename = f"outages_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    # combined_df.to_csv(filename, index=False)
    
    print(f"Data updated successfully at {timestamp}")

# Schedule the scraping job
# Run every 15 minutes
schedule.every(15).minutes.do(scrape_and_update)

# Or run every hour
# schedule.every().hour.do(scrape_and_update)

# Or run at specific times
# schedule.every().day.at("10:00").do(scrape_and_update)

print("Real-time monitoring started...")
print("Press Ctrl+C to stop")

# Keep the script running
# Uncomment below to start continuous monitoring
# while True:
#     schedule.run_pending()
#     time.sleep(1)

ModuleNotFoundError: No module named 'schedule'

In [1]:
# Example: Filter outages affecting hospitals and schools
# This is what you'll need for your project

# Assuming combined_df has your outage data with 'Outage Area' column

def identify_critical_facilities(df):
    """
    Identifies power outages affecting hospitals and schools
    """
    # Keywords to search for hospitals and schools
    hospital_keywords = ['hospital', 'clinic', 'medical', 'health center', 'dispensary']
    school_keywords = ['school', 'college', 'university', 'institute', 'education']
    
    # Create boolean masks
    df['Affects_Hospital'] = df['Outage Area'].str.lower().str.contains(
        '|'.join(hospital_keywords), na=False, case=False
    )
    
    df['Affects_School'] = df['Outage Area'].str.lower().str.contains(
        '|'.join(school_keywords), na=False, case=False
    )
    
    df['Critical_Facility'] = df['Affects_Hospital'] | df['Affects_School']
    
    return df

# Apply to your data
if not combined_df.empty:
    critical_outages = identify_critical_facilities(combined_df.copy())
    
    # Filter only critical facilities
    facilities_affected = critical_outages[critical_outages['Critical_Facility'] == True]
    
    print("="*80)
    print("CRITICAL FACILITIES AFFECTED BY POWER OUTAGES")
    print("="*80)
    print(f"\nTotal affected areas with hospitals/schools: {len(facilities_affected)}")
    
    if len(facilities_affected) > 0:
        print("\nAffected Areas:")
        print(facilities_affected[['Name of Town', 'Outage Area', 'Outage Start Date', 
                                   'Outage End Date (Expected)', 'Affects_Hospital', 
                                   'Affects_School', 'Outage_Type']])
        
        # Save critical facilities data
        facilities_affected.to_csv('critical_facilities_outages.csv', index=False)
        print("\n✓ Critical facilities data saved to 'critical_facilities_outages.csv'")
    else:
        print("\n✓ No hospitals or schools found in outage areas (might need area name matching)")
else:
    print("No data available to analyze")

NameError: name 'combined_df' is not defined

In [7]:
# Production cell: Raipur school/hospital outage risk prediction (next 24h)
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
PROCESSED = ROOT / "processed_outputs"
PROCESSED.mkdir(exist_ok=True)

outage_path = ROOT / "combined_outages.csv"
climate_path = PROCESSED / "environmental_daily_raipur.csv"
facilities_path = PROCESSED / "facilities_master_generated_raipur.csv"
crs_path = PROCESSED / "crs_predictions_raipur_facilities.csv"


def normalize_cols(df):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def find_col(columns, options):
    norm = {c.lower().strip(): c for c in columns}
    for opt in options:
        for k, v in norm.items():
            if opt in k:
                return v
    return None


# 1) Load outage data and keep Raipur rows
if not outage_path.exists():
    raise FileNotFoundError(f"Missing outage file: {outage_path}")

outages = normalize_cols(pd.read_csv(outage_path))

col_town = find_col(outages.columns, ["name of town", "town", "city", "district"])
col_area = find_col(outages.columns, ["outage area", "affected area", "area"])
col_maint = find_col(outages.columns, ["maintenance activity", "feeder", "line"])
col_start_date = find_col(outages.columns, ["outage start date", "start date", "date"])
col_start_time = find_col(outages.columns, ["outage start time", "start time", "time"])

if col_start_date is None:
    raise ValueError("Could not detect outage start date column in combined_outages.csv")

if col_start_time is not None:
    outages["event_ts"] = pd.to_datetime(
        outages[col_start_date].astype(str) + " " + outages[col_start_time].astype(str),
        errors="coerce"
    )
else:
    outages["event_ts"] = pd.to_datetime(outages[col_start_date], errors="coerce")

outages["event_date"] = outages["event_ts"].dt.date
outages = outages.dropna(subset=["event_date"]).copy()

text_for_raipur = pd.Series("", index=outages.index, dtype="object")
if col_town is not None:
    text_for_raipur = text_for_raipur + " " + outages[col_town].astype(str)
if col_area is not None:
    text_for_raipur = text_for_raipur + " " + outages[col_area].astype(str)

raipur_mask = text_for_raipur.str.contains("raipur", case=False, na=False)
outages_raipur = outages[raipur_mask].copy()
if outages_raipur.empty:
    print("Warning: No explicit Raipur rows found in outage feed; using full outage dataset.")
    outages_raipur = outages.copy()

# 2) Detect school/hospital related outage rows
hospital_kw = ["hospital", "clinic", "medical", "health", "nursing"]
school_kw = ["school", "college", "university", "institute", "academy"]

critical_text = pd.Series("", index=outages_raipur.index, dtype="object")
for c in [col_town, col_area, col_maint]:
    if c is not None:
        critical_text = critical_text + " " + outages_raipur[c].astype(str)
critical_text = critical_text.str.lower()

outages_raipur["affects_hospital"] = critical_text.str.contains("|".join(hospital_kw), na=False)
outages_raipur["affects_school"] = critical_text.str.contains("|".join(school_kw), na=False)
outages_raipur["is_critical"] = outages_raipur["affects_hospital"] | outages_raipur["affects_school"]

critical = outages_raipur[outages_raipur["is_critical"]].copy()
if critical.empty:
    print("Warning: No keyword-matched critical outages found; falling back to all Raipur outages for model training.")
    critical = outages_raipur.copy()

# 3) Build daily outage features
outages_raipur["event_date"] = pd.to_datetime(outages_raipur["event_date"])
critical["event_date"] = pd.to_datetime(critical["event_date"])

daily_all = (
    outages_raipur
    .groupby("event_date", as_index=False)
    .agg(outage_events=("event_date", "size"))
    .sort_values("event_date")
)

daily_critical = (
    critical
    .groupby("event_date", as_index=False)
    .agg(
        hospital_events=("affects_hospital", "sum"),
        school_events=("affects_school", "sum")
    )
    .sort_values("event_date")
)

daily_out = daily_all.merge(daily_critical, on="event_date", how="left").fillna(0)
if len(daily_out) < 3:
    print("Warning: Very few outage dates in current feed; model will rely more on climate signal.")

# 4) Load and pivot climate features
if not climate_path.exists():
    raise FileNotFoundError(f"Missing climate file: {climate_path}")

clim = normalize_cols(pd.read_csv(climate_path))
required = {"metric", "date", "mean_value"}
if not required.issubset(set([c.lower() for c in clim.columns])):
    raise ValueError("environmental_daily_raipur.csv must contain metric, date, mean_value columns")

# Resolve exact case-sensitive names
clim_metric = [c for c in clim.columns if c.lower() == "metric"][0]
clim_date = [c for c in clim.columns if c.lower() == "date"][0]
clim_mean = [c for c in clim.columns if c.lower() == "mean_value"][0]

clim[clim_date] = pd.to_datetime(clim[clim_date], errors="coerce")
clim = clim.dropna(subset=[clim_date])

clim_wide = (
    clim.pivot_table(index=clim_date, columns=clim_metric, values=clim_mean, aggfunc="mean")
    .reset_index()
    .rename(columns={clim_date: "event_date"})
)

for c in list(clim_wide.columns):
    if c != "event_date":
        clim_wide = clim_wide.rename(columns={c: f"climate_{c}"})

# 5) Join outage + climate, create next-day target
# Use climate date index as backbone so sparse outage snapshots still produce a usable panel.
model_df = clim_wide.merge(daily_out, on="event_date", how="left").sort_values("event_date")
for c in ["outage_events", "hospital_events", "school_events"]:
    if c not in model_df.columns:
        model_df[c] = 0
model_df[["outage_events", "hospital_events", "school_events"]] = model_df[
    ["outage_events", "hospital_events", "school_events"]
].fillna(0)

model_df["target_outage_next_24h"] = (model_df["outage_events"].shift(-1) > 0).astype(int)
model_df = model_df.iloc[:-1].copy()  # last row has no t+1 label

# Add simple temporal and lag features
model_df["dow"] = model_df["event_date"].dt.dayofweek
model_df["month"] = model_df["event_date"].dt.month
model_df["outage_events_lag1"] = model_df["outage_events"].shift(1)
model_df["outage_events_roll3"] = model_df["outage_events"].rolling(3, min_periods=1).mean()
model_df = model_df.fillna(0)

# 6) Train a lightweight model (with safe fallback)
feature_cols = [
    c for c in model_df.columns
    if c not in ["event_date", "target_outage_next_24h"] and pd.api.types.is_numeric_dtype(model_df[c])
]

latest_features = model_df[feature_cols].iloc[[-1]].copy() if len(model_df) else None
latest_prob = 0.5

if len(model_df) >= 12 and model_df["target_outage_next_24h"].nunique() > 1:
    try:
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler

        X = model_df[feature_cols]
        y = model_df["target_outage_next_24h"]

        clf = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
        ])
        clf.fit(X, y)

        if latest_features is not None:
            latest_prob = float(clf.predict_proba(latest_features)[0, 1])

        model_mode = "logistic_regression"
    except Exception as e:
        print(f"Model fallback triggered: {e}")
        model_mode = "heuristic"
else:
    model_mode = "heuristic"

if model_mode == "heuristic":
    base = 0.12
    recent_out = float(model_df["outage_events"].tail(7).mean()) if len(model_df) else 0.0
    rain_cols = [c for c in feature_cols if "rain" in c.lower()]
    rain_val = float(model_df[rain_cols].tail(7).mean().mean()) if rain_cols and len(model_df) else 0.0
    river_cols = [c for c in feature_cols if "river" in c.lower() or "water" in c.lower()]
    river_val = float(model_df[river_cols].tail(7).mean().mean()) if river_cols and len(model_df) else 0.0
    latest_prob = max(0.05, min(0.95, base + 0.06 * recent_out + 0.0025 * rain_val + 0.001 * river_val))

# 7) Facility-level risk ranking for schools/hospitals in Raipur
if not facilities_path.exists():
    raise FileNotFoundError(f"Missing facilities file: {facilities_path}")

fac = normalize_cols(pd.read_csv(facilities_path))

for needed in ["facility_id", "facility_name", "facility_type", "district"]:
    if needed not in fac.columns:
        raise ValueError(f"facilities_master_generated_raipur.csv missing required column: {needed}")

fac = fac[fac["district"].astype(str).str.upper().str.contains("RAIPUR", na=False)].copy()
fac = fac[fac["facility_type"].astype(str).str.lower().isin(["hospital", "school"])].copy()

# Type multipliers from recent 30-day observed critical outages
max_date = critical["event_date"].max() if len(critical) else pd.Timestamp.now()
recent = critical[critical["event_date"] >= (max_date - pd.Timedelta(days=30))].copy() if len(critical) else critical

h_count = int(recent["affects_hospital"].sum()) if "affects_hospital" in recent.columns else 0
s_count = int(recent["affects_school"].sum()) if "affects_school" in recent.columns else 0
n_total = max(1, h_count + s_count)

type_multiplier = {
    "hospital": 1.0 + (h_count / n_total) * 0.25,
    "school": 1.0 + (s_count / n_total) * 0.25,
}

fac["type_multiplier"] = fac["facility_type"].astype(str).str.lower().map(type_multiplier).fillna(1.0)
fac["pred_outage_prob_24h"] = (latest_prob * fac["type_multiplier"]).clip(0.01, 0.99)

if crs_path.exists():
    crs = normalize_cols(pd.read_csv(crs_path))
    join_cols = ["facility_id", "crs", "risk_level"]
    keep = [c for c in join_cols if c in crs.columns]
    if "facility_id" in keep:
        fac = fac.merge(crs[keep].drop_duplicates(subset=["facility_id"]), on="facility_id", how="left")

if "crs" not in fac.columns:
    fac["crs"] = 0.0

fac["crs"] = pd.to_numeric(fac["crs"], errors="coerce").fillna(0.0).clip(0, 1)
fac["priority_score"] = (0.6 * fac["pred_outage_prob_24h"] + 0.4 * fac["crs"]).clip(0, 1)
fac["predicted_risk_label"] = pd.cut(
    fac["priority_score"],
    bins=[-np.inf, 0.33, 0.66, np.inf],
    labels=["Low", "Medium", "High"]
)

ranked = fac.sort_values(["priority_score", "pred_outage_prob_24h"], ascending=False).copy()

# 8) Save outputs
daily_features_out = PROCESSED / "raipur_daily_model_features.csv"
ranked_out = PROCESSED / "raipur_facility_outage_risk_24h.csv"

model_df.to_csv(daily_features_out, index=False)
ranked.to_csv(ranked_out, index=False)

print("=" * 80)
print("RAIPUR OUTAGE PREDICTION PIPELINE COMPLETED")
print("=" * 80)
print(f"Model mode: {model_mode}")
print(f"Base next-24h outage probability: {latest_prob:.4f}")
print(f"Rows used for model: {len(model_df)}")
print(f"Critical outage rows: {len(critical)}")
print(f"Facilities ranked: {len(ranked)}")
print(f"Saved: {daily_features_out}")
print(f"Saved: {ranked_out}")

display_cols = [
    c for c in [
        "facility_id", "facility_name", "facility_type", "district",
        "pred_outage_prob_24h", "crs", "priority_score", "predicted_risk_label"
    ] if c in ranked.columns
]
print("\nTop 15 facilities by priority score:")
print(ranked[display_cols].head(15))

RAIPUR OUTAGE PREDICTION PIPELINE COMPLETED
Model mode: heuristic
Base next-24h outage probability: 0.4560
Rows used for model: 1446
Critical outage rows: 1
Facilities ranked: 148
Saved: c:\Users\ASUS\Downloads\ThinkX_database\processed_outputs\raipur_daily_model_features.csv
Saved: c:\Users\ASUS\Downloads\ThinkX_database\processed_outputs\raipur_facility_outage_risk_24h.csv

Top 15 facilities by priority score:
         facility_id                                    facility_name  \
4    node/4445117924                            Krishna Public School   
6    node/4445120486                   Holy Heartseducational academy   
7    node/4445126014                                    Mayoor school   
11   node/4445158557              Bharat Mata higher secondary school   
12   node/4445165297                        Dronacharya public school   
13   node/4445176350                           The Radient Way School   
14   node/4445182024               Holy cross senior secondary school   


In [8]:
# Enrich CRS predictions with complete and street addresses for Raipur hospitals/schools
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
PROCESSED = ROOT / "processed_outputs"

crs_path = PROCESSED / "crs_predictions_raipur_facilities.csv"
export_path = ROOT / "export.csv"

if not crs_path.exists():
    raise FileNotFoundError(f"Missing file: {crs_path}")
if not export_path.exists():
    raise FileNotFoundError(f"Missing file: {export_path}")

crs = pd.read_csv(crs_path)
osm = pd.read_csv(export_path)

# Normalize key column from OSM export
id_col = "@id" if "@id" in osm.columns else ("id" if "id" in osm.columns else None)
if id_col is None:
    raise ValueError("export.csv does not contain @id or id column")

# Build address columns with safe fallbacks
for c in ["addr:full", "addr:street", "addr:city", "addr:district", "addr:subdistrict", "addr:housenumber", "addr:postcode", "addr:state"]:
    if c not in osm.columns:
        osm[c] = ""

addr = osm[[
    id_col,
    "addr:full",
    "addr:street",
    "addr:city",
    "addr:district",
    "addr:subdistrict",
    "addr:housenumber",
    "addr:postcode",
    "addr:state",
]].copy()

addr = addr.rename(columns={id_col: "facility_id"})

addr["street_address"] = (
    addr["addr:housenumber"].fillna("").astype(str).str.strip() + " " +
    addr["addr:street"].fillna("").astype(str).str.strip()
).str.strip()

fallback_full = (
    addr["street_address"].replace("", pd.NA).fillna("") + ", " +
    addr["addr:subdistrict"].fillna("").astype(str).str.strip() + ", " +
    addr["addr:district"].fillna("").astype(str).str.strip() + ", " +
    addr["addr:city"].fillna("").astype(str).str.strip() + ", " +
    addr["addr:state"].fillna("").astype(str).str.strip() + " " +
    addr["addr:postcode"].fillna("").astype(str).str.strip()
).str.replace(r"\s+,", ",", regex=True).str.replace(r",\s*,", ",", regex=True).str.strip(" ,")

addr["complete_address"] = addr["addr:full"].fillna("").astype(str).str.strip()
addr.loc[addr["complete_address"].eq(""), "complete_address"] = fallback_full

# Keep only required plus useful raw fields
addr_keep = addr[[
    "facility_id",
    "complete_address",
    "street_address",
    "addr:city",
    "addr:district",
    "addr:subdistrict",
    "addr:postcode",
    "addr:state",
]].drop_duplicates(subset=["facility_id"])

out = crs.merge(addr_keep, on="facility_id", how="left")

# Ensure only Raipur schools/hospitals remain (defensive filter)
out = out[out["district"].astype(str).str.upper().str.contains("RAIPUR", na=False)].copy()
out = out[out["facility_type"].astype(str).str.lower().isin(["hospital", "school"])].copy()

out.to_csv(crs_path, index=False)

print("Address enrichment completed for CRS predictions")
print(f"Updated file: {crs_path}")
print(f"Rows: {len(out)}")
print("Preview:")
print(out[["facility_id", "facility_name", "facility_type", "district", "complete_address", "street_address"]].head(10))

Address enrichment completed for CRS predictions
Updated file: c:\Users\ASUS\Downloads\ThinkX_database\processed_outputs\crs_predictions_raipur_facilities.csv
Rows: 148
Preview:
        facility_id                                    facility_name  \
0   node/6956627669    Raipur Institute Of Medical Sciences Hospital   
1   node/7148125239                        Government hospital arang   
2   node/7148125257                Goverment hospital Gobra Nawapara   
3  node/12276858430                           Campion School, Raipur   
4   node/6965834182                 Jyoti Hospital and Prasuti Griha   
5   node/7148125143                       Government Hospital, Tilda   
6   node/6966792004                     Evangelical Mission Hospital   
7   node/6966522779                                 Mgm Eye Hospital   
8   node/4445255005  Bharatiya vidya bhuvans R.K. sarda vidya mandir   
9   node/7148125120                     Government Hospital Uparwara   

  facility_type district     